<a href="https://colab.research.google.com/github/Desire-in-tech/stock-market-volatility-forecasting/blob/main/notebooks/02_data_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 — Data Model & Test-Driven Development
**BSE Stock Market Volatility Forecasting**

**Goals:**
1. Understand the `data.py` module (`YFinanceAPI`, `SQLRepository`)
2. Set up a SQLite database to persist stock data
3. Practice TDD: write tests *before* (or alongside) the implementation
4. Insert and retrieve BSE data using `SQLRepository`
5. Verify the ETL pipeline end-to-end

In [1]:
!git clone https://github.com/Desire-in-tech/stock-market-volatility-forecasting.git

Cloning into 'stock-market-volatility-forecasting'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 24 (delta 2), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 264.12 KiB | 6.29 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [2]:
%cd stock-market-volatility-forecasting

/content/stock-market-volatility-forecasting


In [3]:
!ls src

data.py  main.py  model.py


## 1. Setup

In [4]:
import sys
import os
import sqlite3
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# Add src/ to path so we can import data.py and model.py
sys.path.insert(0, 'src')

from data import YFinanceAPI, SQLRepository, get_stock_data

print('Imports OK')

# Paths
DB_PATH = os.path.join('..', 'database', 'stock_data.db')
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
print(f'Database path: {os.path.abspath(DB_PATH)}')

Imports OK
Database path: /content/database/stock_data.db


## 2. Understanding `data.py`

Our `src/data.py` provides three things:

| Class / Function | Responsibility |
|---|---|
| `YFinanceAPI` | Downloads OHLCV data from Yahoo Finance |
| `SQLRepository` | Reads/writes DataFrames to SQLite |
| `get_stock_data()` | One-call convenience wrapper |


In [5]:
# ── YFinanceAPI ──────────────────────────────────────────────────────────────
api = YFinanceAPI()

# Fetch SENSEX data
df_sensex = api.get_daily_data('^BSESN', start='2020-01-01', end='2024-12-31')

print(f'Rows fetched : {len(df_sensex):,}')
print(f'Columns      : {list(df_sensex.columns)}')
print(f'Index type   : {type(df_sensex.index).__name__}')
print(f'Null values  : {df_sensex.isnull().sum().sum()}')
df_sensex.head()

Rows fetched : 1,232
Columns      : ['Open', 'High', 'Low', 'Close', 'Volume', 'returns']
Index type   : DatetimeIndex
Null values  : 0


Price,Open,High,Low,Close,Volume,returns
Date,,,,,,
2020-01-03,41634.511719,41636.179688,41348.679688,41464.609375,8900,-0.389249
2020-01-06,41378.339844,41378.339844,40613.960938,40676.628906,8700,-1.900369
2020-01-07,40983.039062,41230.140625,40727.371094,40869.468750,11900,0.474080
2020-01-08,40574.828125,40866.359375,40476.550781,40817.738281,18200,-0.126575
2020-01-09,41216.671875,41482.121094,41175.718750,41452.351562,7800,1.554749


## 3. SQLite Database Setup

We use SQLite with `SQLRepository` to store our clean data.

In [6]:
# Create SQLite connection and repository
connection = sqlite3.connect(DB_PATH)
repo = SQLRepository(connection=connection)

print(f'Connected to: {DB_PATH}')
print(f'SQLRepository instance: {repo}')

Connected to: ../database/stock_data.db
SQLRepository instance: <data.SQLRepository object at 0x7d36c5d0cb30>


## 4. Insert Data into SQLite

In [7]:
result = repo.insert_table(
    table_name='^BSESN',
    records=df_sensex,
    if_exists='replace'
)
print('Insert result:', result)

Insert result: {'transaction_successful': True, 'records_inserted': 1232}


In [8]:
# Insert a few more stocks
STOCKS_TO_LOAD = [
    'RELIANCE.NS',
    'TCS.NS',
    'INFY.NS',
    'HDFCBANK.NS',
]

for ticker in STOCKS_TO_LOAD:
    data = api.get_daily_data(ticker, start='2015-01-01', end='2024-12-31')
    res = repo.insert_table(table_name=ticker, records=data, if_exists='replace')
    print(f'{ticker:<20}  →  {res["records_inserted"]:>4} rows inserted')

RELIANCE.NS           →  2465 rows inserted
TCS.NS                →  2465 rows inserted
INFY.NS               →  2465 rows inserted
HDFCBANK.NS           →  2465 rows inserted


## 5. Read Data Back from SQLite

In [9]:
# Read all SENSEX rows
df_from_db = repo.read_table('^BSESN')
print(f'Read {len(df_from_db):,} rows from DB')
print(f'Columns: {list(df_from_db.columns)}')
df_from_db.tail()

Read 1,232 rows from DB
Columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'returns']


,Open,High,Low,Close,Volume,returns
Date,,,,,,
2024-12-23,78488.640625,78918.117188,78189.187500,78540.171875,10300,0.638862
2024-12-24,78707.367188,78877.359375,78397.789062,78472.867188,6200,-0.085695
2024-12-26,78557.281250,78898.367188,78173.382812,78472.476562,5600,-0.000498
2024-12-27,78607.617188,79043.148438,78598.546875,78699.070312,7700,0.288756
2024-12-30,78637.578125,79092.703125,78077.132812,78248.132812,8900,-0.572990


In [10]:
# Read only the most recent 500 rows
df_recent = repo.read_table('^BSESN', limit=500)
print(f'Read {len(df_recent):,} rows (limited to 500)')
print(f'Date range: {df_recent.index.min().date()}  →  {df_recent.index.max().date()}')

Read 500 rows (limited to 500)
Date range: 2022-12-16  →  2024-12-30


## 6. Test-Driven Development (TDD)

TDD means: **write the test → run it (expect failure) → write code to make it pass → refactor**.

We write inline assertion-style tests here; the full pytest suite lives in `tests/`.

In [11]:
def assert_equal(actual, expected, label=''):
    if actual == expected:
        print(f'  PASS  {label}')
    else:
        print(f'  FAIL  {label}  (got {actual!r}, expected {expected!r})')

def assert_true(condition, label=''):
    if condition:
        print(f'  PASS  {label}')
    else:
        print(f'  FAIL  {label}')

print('=== YFinanceAPI tests ===')

sample = api.get_daily_data('^BSESN', '2023-01-01', '2023-06-30')

assert_true(isinstance(sample, pd.DataFrame),
            'get_daily_data returns a DataFrame')
assert_true(isinstance(sample.index, pd.DatetimeIndex),
            'Index is DatetimeIndex')
assert_true('returns' in sample.columns,
            'returns column present')
assert_equal(sample['returns'].isna().sum(), 0,
             'No NaN in returns')
assert_true(len(sample) > 0,
            'Non-empty DataFrame returned')

=== YFinanceAPI tests ===
  PASS  get_daily_data returns a DataFrame
  PASS  Index is DatetimeIndex
  PASS  returns column present
  PASS  No NaN in returns
  PASS  Non-empty DataFrame returned


In [12]:
# ── TDD: SQLRepository ─
print('=== SQLRepository tests ===')

# Use an in-memory DB so tests don't pollute the real DB
test_conn = sqlite3.connect(':memory:')
test_repo = SQLRepository(connection=test_conn)

test_df = pd.DataFrame(
    {'Open': [100.0, 101.0], 'Close': [101.0, 102.0], 'returns': [1.0, 0.99]},
    index=pd.to_datetime(['2023-01-02', '2023-01-03'])
)
test_df.index.name = 'Date'

insert_result = test_repo.insert_table('TEST', test_df, if_exists='replace')
assert_true(insert_result['transaction_successful'],
            'insert_table returns success')

read_back = test_repo.read_table('TEST')
assert_true(isinstance(read_back, pd.DataFrame),
            'read_table returns DataFrame')
assert_equal(len(read_back), 2,
             'Correct number of rows read back')

limited = test_repo.read_table('TEST', limit=1)
assert_equal(len(limited), 1,
             'limit parameter works')

assert_true(test_repo.table_exists('TEST'),
            'table_exists returns True for existing table')
assert_true(not test_repo.table_exists('NONEXISTENT'),
            'table_exists returns False for missing table')

test_conn.close()

=== SQLRepository tests ===
  PASS  insert_table returns success
  PASS  read_table returns DataFrame
  PASS  Correct number of rows read back
  PASS  limit parameter works
  PASS  table_exists returns True for existing table
  PASS  table_exists returns False for missing table


## 7. Verify the Full ETL Pipeline

In [13]:
# Full pipeline test: fetch → insert → read → compare
TICKER = 'WIPRO.NS'
START, END = '2022-01-01', '2024-01-01'

print(f'Step 1: Fetch {TICKER}...')
fetched = api.get_daily_data(TICKER, START, END)
print(f'  Fetched {len(fetched):,} rows')

print('Step 2: Insert into DB...')
result = repo.insert_table(TICKER, fetched, if_exists='replace')
print(f'  {result}')

print('Step 3: Read back from DB...')
loaded = repo.read_table(TICKER)
print(f'  Read {len(loaded):,} rows')

print('Step 4: Verify round-trip...')
close_match = (fetched['Close'].round(4) == loaded['Close'].round(4)).all()
print(f'  Close prices match: {close_match}')
print('\nETL pipeline is working correctly!')

Step 1: Fetch WIPRO.NS...
  Fetched 492 rows
Step 2: Insert into DB...
  {'transaction_successful': True, 'records_inserted': 492}
Step 3: Read back from DB...
  Read 492 rows
Step 4: Verify round-trip...
  Close prices match: True

ETL pipeline is working correctly!


## 8. Inspect the Database

In [14]:
# List all tables in the database
tables = pd.read_sql(
    "SELECT name, type FROM sqlite_master WHERE type='table'",
    con=connection
)
print(f'Tables in {DB_PATH}:')
print(tables.to_string(index=False))

Tables in ../database/stock_data.db:
       name  type
     ^BSESN table
RELIANCE.NS table
     TCS.NS table
    INFY.NS table
HDFCBANK.NS table
   WIPRO.NS table


In [15]:
# Row count per table
for tbl in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM '{tbl}'", con=connection).iloc[0]['n']
    print(f'  {tbl:<25}  {count:>5} rows')

  ^BSESN                      1232 rows
  RELIANCE.NS                 2465 rows
  TCS.NS                      2465 rows
  INFY.NS                     2465 rows
  HDFCBANK.NS                 2465 rows
  WIPRO.NS                     492 rows


In [16]:
# Clean up connection
connection.close()
print('Connection closed.')

Connection closed.


## Summary

- `YFinanceAPI.get_daily_data()` fetches and cleans BSE/NSE data reliably.
- `SQLRepository` persists and retrieves DataFrames to/from SQLite with a clean interface.
- All tests pass — the ETL layer is solid.
- Full pytest suite: `pytest tests/test_data.py -v`